# MERT vs AST — VocalSet 창법 분류 비교

**시작 전:** 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택 후 `런타임 → 모두 실행`.

전체 소요: 다운로드 5~15분 + 임베딩 추출 모델당 10~30분(T4 기준 추정) + 평가 수 분.

In [ ]:
!nvidia-smi -L || echo "GPU 없음: 런타임 유형을 T4 GPU로 바꾸세요 (CPU로도 되지만 매우 느림)"
!rm -rf songlab && git clone -q -b claude/spectrogram-reading-model-itsg91 https://github.com/tfvs2023-hub/songlab
%cd songlab
!pip install -q -r experiments/vocal_probe/requirements.txt

## 1. VocalSet 다운로드 (Zenodo, 약 수 GB)

In [ ]:
import json, pathlib, subprocess, urllib.request

DATA = pathlib.Path("/content/vocalset")
DATA.mkdir(exist_ok=True)
zips = list(DATA.glob("*.zip"))
if not zips:
    # VocalSet 버전별 Zenodo 레코드를 차례로 조회해 zip 파일을 찾는다
    for record in ("1442513", "1492453", "1193957"):
        try:
            meta = json.load(urllib.request.urlopen(f"https://zenodo.org/api/records/{record}"))
        except Exception as e:
            print(record, "조회 실패:", e); continue
        files = [f for f in meta.get("files", []) if f["key"].lower().endswith(".zip")]
        if files:
            f = files[0]
            print(f"record {record}: {f['key']} ({f['size']/1e9:.1f} GB) 다운로드")
            subprocess.run(["wget", "-q", "--show-progress", "-O", str(DATA / f["key"]), f["links"]["self"]], check=True)
            break
    zips = list(DATA.glob("*.zip"))
assert zips, "자동 다운로드 실패: https://doi.org/10.5281/zenodo.1442513 에서 zip을 받아 /content/vocalset 에 올리세요"
!cd /content/vocalset && unzip -q -n "*.zip"

fulls = [p for p in DATA.rglob("FULL") if p.is_dir()]
DATA_ROOT = str(fulls[0] if fulls else DATA)
print("DATA_ROOT =", DATA_ROOT)

## 2. 임베딩 추출

In [ ]:
!python experiments/vocal_probe/extract.py --model mert --data-root "$DATA_ROOT"

In [ ]:
!python experiments/vocal_probe/extract.py --model ast --data-root "$DATA_ROOT"

MERT 로딩 에러(`ImportError` 등)가 나면 아래 셀 주석을 풀어 transformers를 MERT 공식 버전으로 맞춘 뒤,
`런타임 → 세션 다시 시작` 후 MERT 셀부터 다시 실행하세요.

In [ ]:
# !pip install -q transformers==4.38.2

## 3. 비교 평가 (가수 단위 5-fold)

In [ ]:
!python experiments/vocal_probe/evaluate.py outputs/vocal_probe/mert.npz outputs/vocal_probe/ast.npz --csv outputs/vocal_probe/scores.csv

## 4. 결과 저장
출력된 Summary를 복사해 붙여주시거나 아래로 받은 파일을 공유해 주세요.

In [ ]:
from google.colab import files
files.download("outputs/vocal_probe/scores.csv")